In [23]:
import pickle

# run the model
with open('../xgb_model_withgpa.pkl', 'rb') as file:
    xgb_model = pickle.load(file)



In [24]:
import pandas as pd
import numpy as np

In [25]:
xgb_input = pd.read_csv("../x.csv")
xgb_input.head(10)

,Unnamed: 0,anxious,calm,conventional,critical,dependable,disorganized,enthusiastic,experiences,reserved,...,hour,exercise,walk,have,schedule,sleep_hours,sleep_quality,sleepiness,number,gpa_all
0,0,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
1,1,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
2,2,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
3,3,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
4,4,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
5,5,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
6,6,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
7,7,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
8,8,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
9,9,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505


## Noise experiment on highly relevent variables

In [26]:
print("sadornot distinct values:", xgb_input['sadornot'].unique())
print("experience distinct values:", xgb_input['experience'].unique())
print("gpa_all distinct values:", xgb_input['gpa_all'].unique())


sadornot distinct values: [1. 2.]
experience distinct values: [2 3 1 4 5]
gpa_all distinct values: [3.505 3.029 3.474 3.705 3.667 3.245 3.293 3.373 3.476 3.947 3.719 3.826
 2.815 3.79  3.625 2.4   3.519]


In [27]:
# add noise to variables: sadornot
# Randomly add 1, subtract 1, or make no change for each sample.

import numpy as np

noisy_input = xgb_input.copy()

sadornot_noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0])

noisy_input['sadornot'] = noisy_input['sadornot'] + sadornot_noise
noisy_input['sadornot'] = noisy_input['sadornot'].clip(lower=1, upper=2)

print(noisy_input['sadornot'].value_counts().sort_index())



sadornot
1.0    4699
2.0    4304
Name: count, dtype: int64


In [28]:
# add noise to variable: experience
# apply normal distribution to choose the reasonable noise to add on the original experience column

experience_noise = np.random.normal(loc=0, scale=1, size=noisy_input.shape[0])

noisy_input['experience'] = noisy_input['experience'] + experience_noise
noisy_input['experience'] = noisy_input['experience'].round().astype(int)
noisy_input['experience'] = noisy_input['experience'].clip(lower=1, upper=5)

print(noisy_input['experience'].head(10))



0    3
1    2
2    1
3    1
4    1
5    1
6    3
7    2
8    2
9    2
Name: experience, dtype: int64


In [29]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_input.shape[0])
noisy_input['gpa_all'] = noisy_input['gpa_all'] + gpa_noise
print(noisy_input['gpa_all'].head(10))

0    3.441408
1    3.536808
2    3.465023
3    3.461865
4    3.510226
5    3.454848
6    3.476997
7    3.562407
8    3.624640
9    3.592383
Name: gpa_all, dtype: float64


In [30]:
# re-run the xgboost model
noisy_input = noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

y_pred_noisy = xgb_model.predict(noisy_input)


In [31]:
from sklearn.metrics import mean_squared_error
y_data = pd.read_csv("../y.csv")

y_true = y_data['total_score']
mse_noisy = mean_squared_error(y_true, y_pred_noisy)

print("MSE after adding noise on highly relevant variables:", mse_noisy)


MSE after adding noise on highly relevant variables: 17.30774921018608


In [32]:
from sklearn.metrics import r2_score
r2_noisy = r2_score(y_true, y_pred_noisy)
print("R² after adding noise on highly relevant variables:", r2_noisy)

R² after adding noise on highly relevant variables: 0.4577192747528649


## Noise experiment on less relevent variables

In [33]:
print("has_negative_text distinct values:", xgb_input['has_negative_text'].unique())
print("schedule distinct values:", xgb_input['schedule'].unique())
print("have distinct values:", xgb_input['have'].unique())

has_negative_text distinct values: [1. 0.]
schedule distinct values: [1. 2.]
have distinct values: [1. 2.]


In [34]:
import numpy as np

# 复制数据
less_relevent_noisy_input = xgb_input.copy()

# 定义要扰动的变量及其合法范围（clip用）
binary_features = {
    'has_negative_text': (0, 1),
    'schedule': (1, 2),
    'have': (1, 2)
}

# 遍历并加噪
for feature, (min_val, max_val) in binary_features.items():
    noise = np.random.choice([-1, 0, 1], size=less_relevent_noisy_input.shape[0])
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature] + noise
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature].clip(lower=min_val, upper=max_val)

# 查看结果
print(less_relevent_noisy_input[list(binary_features.keys())].head(10))


   has_negative_text  schedule  have
0                0.0       1.0   1.0
1                0.0       1.0   2.0
2                1.0       1.0   1.0
3                0.0       2.0   1.0
4                1.0       1.0   2.0
5                1.0       1.0   1.0
6                0.0       1.0   1.0
7                1.0       1.0   1.0
8                1.0       1.0   2.0
9                0.0       1.0   1.0


In [35]:
# re-run the xgboost model
less_relevent_noisy_input = xgb_input.copy()

less_relevent_noisy_input = less_relevent_noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

less_relevent_y_pred_noisy = xgb_model.predict(less_relevent_noisy_input)



In [36]:
less_relevent_mse_noisy = mean_squared_error(y_true, less_relevent_y_pred_noisy)

print("MSE after adding noise for less relevant variables:", less_relevent_mse_noisy)

MSE after adding noise for less relevant variables: 3.2133332959736576


In [37]:
less_relevent_r2_noisy = r2_score(y_true, less_relevent_y_pred_noisy)
print("R² after adding noise on less relevant variables:", less_relevent_r2_noisy)

R² after adding noise on less relevant variables: 0.8993208944132471


| Condition                          | MSE       | RMSE (√MSE) | R²      |
|:----------------------------------|:----------|:------------|:--------|
| No Noise                          | 3.3773    | 1.837       | 0.8963  |
| Noise on Top-3 Important Features | 17.3077   | 4.159       | 0.4577  |
| Noise on Bottom-3 Important Features   | 3.2133    | 1.791       | 0.8993  |

